# 2. Exploración y calidad

Identificaremos nulos, duplicados, tipos incorrectos y valores inconsistentes.


In [14]:
from pathlib import Path
import sys

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))


In [16]:
import importlib
from src.extraccion import MongoDBExtractor
import src.calidad

# Forzamos a Python a leer la versión más reciente del archivo modificado
importlib.reload(src.calidad)
from src.calidad import crear_reporte_calidad

# Creamos la instancia y ejecutamos
extractor_ = MongoDBExtractor()
extractor_.connect()

fuentes = extractor_.extract_all_collections()
reporte = crear_reporte_calidad(fuentes)

#tiempo estimado de espera 10 minutos o menos dependiendo de la lectura de los datos 
display(reporte)

,fuente,filas,columnas,valores_nulos,filas_duplicadas_exactas
0,listing,19187,79,65366,0
1,reviews,527731,7,1,0
2,calendar,7003255,6,0,0


# Valores nulos 
listing:Acumula un total de 65,366 valores nulos distribuidos a lo largo de sus 79 columnas. Estos faltantes se concentran principalmente en atributos descriptivos secundarios, amenidades opcionales o políticas secundarias que los anfitriones omiten al registrar sus alojamientos.
reviews y calendar :Presentan una completitud casi perfecta, con apenas 1 valor nulo en reviews y 0 nulos en calendar, lo que indica una estructura sumamente sólida en las series de disponibilidad y comen

# valores duplicados 
El análisis de duplicados exactos (previa normalización de estructuras complejas anidadas) arrojó 0 filas duplicadas en todas las colecciones (listing, reviews y calendar).

Al no existir registros completamente repetidos, no es necesario aplicar un proceso de eliminación de duplicados en la fase de transformación principal, garantizando que cada documento en MongoDB posee una identidad única (clave primaria basada en _id o identificadores de alojamiento).

# posibles valores atipicos 

-minimum_nights :Presenta un valor máximo de 32 noches, lo cual se alinea con restricciones de estadías de mediano plazo permitidas en la plataforma, aunque el 75% de los alojamientos exigen 3 noches o menos. No requiere una eliminación drástica, pero sí acotación si se buscan estancias turísticas estándar.

availability_365:Sus valores oscilan de forma lógica entre 0 y 365 días, reflejando tanto alojamientos totalmente inactivos/ocupados como aquellos disponibles todo el año. No presenta anomalías lógicas que requieran exclusión.